In [1]:
import pandas as pd
stations = pd.read_csv("https://data.ny.gov/resource/5f5g-n3cz.csv")
stations.head(3)

,complex_id,is_complex,number_of_stations_in_complex,stop_name,display_name,constituent_station_names,station_ids,gtfs_stop_ids,borough,cbd,daytime_routes,structure_type,latitude,longitude,ada,ada_notes
0,398,False,1,77 St,77 St (6),77 St,398,627,M,False,6,Subway,40.773620,-73.959874,0,NaN
1,399,False,1,68 St-Hunter College,68 St-Hunter College (6),68 St-Hunter College,399,628,M,False,6,Subway,40.768141,-73.963870,1,NaN
2,403,False,1,33 St,33 St (6),33 St,403,632,M,True,6,Subway,40.746081,-73.982076,0,NaN


In [2]:
stations[["stop_name","borough"]].head(3)

,stop_name,borough
0,77 St,M
1,68 St-Hunter College,M
2,33 St,M


In [14]:
import sqlite3
conn=sqlite3.connect("memory")
borough=stations[["borough"]]
borough.to_sql("stations",conn,index=False,if_exists="replace")
query="select borough,COUNT(*) FROM stations group by borough"
pd.read_sql_query(query,conn)

,borough,COUNT(*)
0,Bk,156
1,Bx,68
2,M,121
3,Q,79
4,SI,21


In [26]:
station_data=stations[["borough","ada"]]
station_data.to_sql("station_access",conn,index=False,if_exists="replace")
query="SELECT borough,COUNT(*) AS total,COUNT(CASE WHEN ada=1 THEN 1 END) AS accessible FROM station_access GROUP BY borough"
pd.read_sql_query(query,conn)

,borough,total,accessible
0,Bk,156,35
1,Bx,68,21
2,M,121,40
3,Q,79,24
4,SI,21,6


In [33]:
result=pd.read_sql_query(query,conn)
result.assign(accessible_pct=(result["accessible"]/result["total"]*100).round(1))

,borough,total,accessible,accessible_pct
0,Bk,156,35,22.4
1,Bx,68,21,30.9
2,M,121,40,33.1
3,Q,79,24,30.4
4,SI,21,6,28.6
